加载清洗图片

In [ ]:
import os
import cv2
import tensorflow as tf
import numpy as np
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import RandomOverSampler
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.preprocessing import LabelEncoder
from collections import Counter
from tqdm import tqdm

DATASET_DIR = r'data\color-tom' #这里如果不行的话你们把引号里改成完整的文件路径:D
IMAGE_SIZE = (224, 224)

def load_images_and_labels(base_dir):
    data, labels = [], []
    classes = sorted(os.listdir(base_dir))
    for class_name in classes:
        class_dir = os.path.join(base_dir, class_name)
        if not os.path.isdir(class_dir):
            continue
        for file in os.listdir(class_dir):
            path = os.path.join(class_dir, file)
            try:
                img = cv2.imread(path)
                if img is None:
                    continue
                img = cv2.resize(img, IMAGE_SIZE)
                data.append(img)
                labels.append(class_name)
            except:
                continue
    return np.array(data), np.array(labels)

data, labels = load_images_and_labels(DATASET_DIR)
print("原始类别统计:", Counter(labels))

原始类别统计: Counter({np.str_('Tomato___Tomato_Yellow_Leaf_Curl_Virus'): 5357, np.str_('Tomato___Bacterial_spot'): 2127, np.str_('Tomato___Late_blight'): 1909, np.str_('Tomato___Septoria_leaf_spot'): 1771, np.str_('Tomato___Spider_mites Two-spotted_spider_mite'): 1676, np.str_('Tomato___healthy'): 1591, np.str_('Tomato___Target_Spot'): 1404, np.str_('Tomato___Early_blight'): 1000, np.str_('Tomato___Leaf_Mold'): 952, np.str_('Tomato___Tomato_mosaic_virus'): 373})


过采样


In [7]:
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler
from collections import Counter
import numpy as np

target_num = 1000

under_sampling_strategy = {k: target_num for k, v in label_counts.items() if v > target_num}
rus = RandomUnderSampler(sampling_strategy=under_sampling_strategy)
data_rus, labels_rus = rus.fit_resample(data.reshape(len(data), -1), labels_encoded)

label_counts_rus = Counter(labels_rus)

over_sampling_strategy = {k: target_num for k, v in label_counts_rus.items() if v < target_num}
ros = RandomOverSampler(sampling_strategy=over_sampling_strategy)
data_final, labels_final = ros.fit_resample(data_rus, labels_rus)

data_final = data_final.reshape(-1, IMAGE_SIZE[0], IMAGE_SIZE[1], 3)
labels_final = le.inverse_transform(labels_final)

print("最终类别统计:", Counter(labels_final))


最终类别统计: Counter({np.str_('Tomato___Bacterial_spot'): 1000, np.str_('Tomato___Early_blight'): 1000, np.str_('Tomato___Late_blight'): 1000, np.str_('Tomato___Leaf_Mold'): 1000, np.str_('Tomato___Septoria_leaf_spot'): 1000, np.str_('Tomato___Spider_mites Two-spotted_spider_mite'): 1000, np.str_('Tomato___Target_Spot'): 1000, np.str_('Tomato___Tomato_Yellow_Leaf_Curl_Virus'): 1000, np.str_('Tomato___Tomato_mosaic_virus'): 1000, np.str_('Tomato___healthy'): 1000})


图片增强与划分

In [10]:
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

def augment_images(images, labels, augment_times=2):
    augmented_data = []
    augmented_labels = []
    for i in range(len(images)):
        img = images[i]
        label = labels[i]
        img = img.reshape((1,) + img.shape)
        aug_iter = datagen.flow(img, batch_size=1)
        for _ in range(augment_times):
            aug_img = next(aug_iter)[0].astype(np.uint8)
            augmented_data.append(aug_img)
            augmented_labels.append(label)
    return np.array(augmented_data), np.array(augmented_labels)


aug_data, aug_labels = augment_images(data_final, labels_final, augment_times=2)
all_data = np.concatenate([data_final, aug_data])
all_labels = np.concatenate([labels_final, aug_labels])

X_train, X_test, y_train, y_test = train_test_split(
    all_data, all_labels, test_size=0.2, random_state=42, stratify=all_labels
)

保存tfreocrd


In [11]:
def _bytes_feature(value):
    return tf.train.Feature(bytes_list=tf.train.BytesList(value=[value]))

def _int64_feature(value):
    return tf.train.Feature(int64_list=tf.train.Int64List(value=[value]))

def image_example(image, label):
    feature = {
        'image': _bytes_feature(tf.io.encode_jpeg(image).numpy()),
        'label': _int64_feature(label),
    }
    return tf.train.Example(features=tf.train.Features(feature=feature))

def write_tfrecord(filename, images, labels):
    with tf.io.TFRecordWriter(filename) as writer:
        for i in tqdm(range(len(images))):
            example = image_example(tf.convert_to_tensor(images[i]), le.transform([labels[i]])[0])
            writer.write(example.SerializeToString())

write_tfrecord('train.tfrecord', X_train, y_train)
write_tfrecord('test.tfrecord', X_test, y_test)
print("TFRecord 文件生成完毕。")


100%|██████████| 6000/6000 [00:02<00:00, 2578.19it/s]

TFRecord 文件生成完毕。
